# Filament chirality baselines

This notebook establishes baseline accuracy for predicting filament chirality (0 = dextral, 1 = sinistral) from the event catalog, before comparing against Surya.

It does **not** use the PyTorch/Lightning training-loop pattern from **_1_baseline_template.ipynb_** (regression model, metrics, trainer) — neither baseline here trains a model, so that machinery isn't needed. Refer back to that notebook if you want to incorporate a trainable baseline model later.

## Load configuration, scalers, and the dataset

`train_data_path`/`valid_data_path` are overridden here in-memory only — `config_script.yaml` still points at the generic shared Surya indices, which don't cover 2012 (9 of these 11 events fall in 2012). This points instead at the small index built specifically for this catalog (see `data/surya_index_filament_events.csv`).

Neither baseline needs Surya imagery, so the dataset is built with `return_surya_stack=False`.

In [7]:
import sys
sys.path.append("../../")

import numpy as np

from downstream_apps.filament_kyle.configs import load_filament_config
from downstream_apps.filament_kyle.datasets.filament_dataset import FilamentDataset
from workshop_infrastructure.assets import ensure_assets
from workshop_infrastructure.datasets.builders import build_helio_dataloaders
from workshop_infrastructure.utils import build_scalers

cfg = load_filament_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")

ensure_assets(cfg, which=["scalers"])
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")

cfg.data.train_data_path = "./data/surya_index_filament_events.csv"
cfg.data.valid_data_path = "./data/surya_index_filament_events.csv"

baseline_train_loader, baseline_val_loader = build_helio_dataloaders(
    cfg,
    FilamentDataset,
    scalers=scalers,
    num_workers=0,
    return_surya_stack=False,   # neither baseline looks at the Surya imagery
    filament_index_path=cfg.data.filament_index_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
)

baseline_dataset = baseline_train_loader.dataset
n_events = len(baseline_dataset)
labels = np.array([baseline_dataset[i]["forecast"] for i in range(n_events)])
hemispheres = np.array([baseline_dataset[i]["hemisphere"] for i in range(n_events)])

print(f"{n_events} catalog events matched to a Surya frame")
print(f"Chirality counts  — dextral (0): {(labels == 0).sum()}, sinistral (1): {(labels == 1).sum()}")
print(f"Hemisphere counts — south (0): {(hemispheres == 0).sum()}, north (1): {(hemispheres == 1).sum()}")


Loaded config for job: filament_characterization
Loaded scalers for 13 channels.
11 catalog events matched to a Surya frame
Chirality counts  — dextral (0): 9, sinistral (1): 2
Hemisphere counts — south (0): 4, north (1): 7


## Baselines: coin-flip and calibrated Martin's Rule

Before comparing against Surya, we need a baseline number worth beating. The classic **Martin's Rule** (hemisphere predicts chirality, ~90% in the textbook) is *not* used directly here: this catalog was deliberately curated to include events that violate it, so scoring against the textbook rate would be an unfair comparison point — it could make Surya look artificially better or worse depending on how the exceptions happen to land.

Instead we use two baselines that don't assume the textbook rate:

1. **Coin-flip** — random 50/50 guessing. With only 11 (soon 45) events, a single draw is too noisy to be a stable number, so we run a Monte Carlo simulation and report the resulting accuracy distribution.
2. **Calibrated Martin's Rule** — instead of assuming ~90%, estimate `P(chirality | hemisphere)` directly from this catalog. Evaluated via **leave-one-out cross-validation**: for each event, the hemisphere-conditioned rate is estimated from the other events only, so no event is ever used to predict itself.

### Coin-flip baseline (Monte Carlo)

A single set of random 50/50 guesses over 11 events is too noisy to be a meaningful number on its own — one lucky draw could land at 70%+ accuracy by chance. Instead we simulate many trials and report the resulting distribution; the mean is the number to compare Surya against, and the spread shows how much of any single accuracy score is just noise at this sample size.

In [8]:
rng = np.random.default_rng(42)
n_trials = 100_000

coin_flip_predictions = rng.integers(0, 2, size=(n_trials, n_events))
coin_flip_accuracies = (coin_flip_predictions == labels).mean(axis=1)

print(f"Coin-flip baseline over {n_trials:,} trials on {n_events} events:")
print(f"  mean accuracy:       {coin_flip_accuracies.mean():.3f}")
print(f"  std accuracy:        {coin_flip_accuracies.std():.3f}")
print(f"  5th-95th percentile: {np.percentile(coin_flip_accuracies, 5):.3f} - {np.percentile(coin_flip_accuracies, 95):.3f}")


Coin-flip baseline over 100,000 trials on 11 events:
  mean accuracy:       0.500
  std accuracy:        0.150
  5th-95th percentile: 0.273 - 0.727


### Calibrated Martin's Rule (leave-one-out)

Rather than assume the textbook ~90% hemisphere-to-chirality accuracy, we estimate it from this catalog directly: for each event, take the majority chirality among all *other* events sharing its hemisphere, and predict that majority label. This is leave-one-out — an event never contributes to its own prediction, so the accuracy isn't inflated by fitting and scoring on the same data. Ties, or a fold where no other event shares the held-out hemisphere, fall back to the overall majority chirality among the remaining events.

In [9]:
def majority_label(values: np.ndarray) -> float:
    """Majority vote among 0/1 labels; ties resolve to dextral (0)."""
    return 1.0 if values.mean() > 0.5 else 0.0


def loo_calibrated_martins_rule(labels: np.ndarray, hemispheres: np.ndarray) -> np.ndarray:
    n = len(labels)
    predictions = np.empty(n)
    for i in range(n):
        others = np.arange(n) != i
        train_labels, train_hemispheres = labels[others], hemispheres[others]
        same_hemisphere = train_hemispheres == hemispheres[i]
        # Fall back to the overall majority class if no other event shares this
        # hemisphere in the held-out fold.
        pool = train_labels[same_hemisphere] if same_hemisphere.any() else train_labels
        predictions[i] = majority_label(pool)
    return predictions


martins_rule_predictions = loo_calibrated_martins_rule(labels, hemispheres)
martins_rule_correct = martins_rule_predictions == labels
martins_rule_accuracy = martins_rule_correct.mean()

print(f"Calibrated Martin's Rule (leave-one-out) accuracy: "
      f"{martins_rule_accuracy:.3f} ({int(martins_rule_correct.sum())}/{n_events})")


Calibrated Martin's Rule (leave-one-out) accuracy: 0.818 (9/11)


### Baseline summary

These are the two numbers Surya needs to beat — not the textbook Martin's Rule accuracy, which this catalog is specifically designed to violate in places.

In [10]:
print("Baseline summary")
print(f"  Coin-flip (Monte Carlo mean):    {coin_flip_accuracies.mean():.3f}")
print(f"  Calibrated Martin's Rule (LOO):  {martins_rule_accuracy:.3f}")


Baseline summary
  Coin-flip (Monte Carlo mean):    0.500
  Calibrated Martin's Rule (LOO):  0.818
